<a href="https://colab.research.google.com/github/Dipto1971/Statistical-Analysis-Data-Science/blob/main/Assignment/EDA_Automation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Automating Univariate Statistics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def univariate(df):
  import pandas as pd
  import numpy as np
  import matplotlib.pyplot as plt
  import seaborn as sns

  output_df = pd.DataFrame(columns = ['type', 'count', 'missing', 'unique', 'mode', 'min', 'Q1',
                                      'median', 'Q3', 'max', 'mean', 'std', 'skewness', 'kurtosis'])

  for col in df:
    # Calculate metrics that apply to all dtypes
    dtype = df[col].dtype
    count = df[col].count()
    missing = df[col].isnull().sum()
    unique = df[col].nunique()
    mode = df[col].mode()[0]

    if pd.api.types.is_numeric_dtype(df[col]):
      min = df[col].min()
      Q1 = df[col].quantile(0.25)
      median = df[col].median()
      Q3 = df[col].quantile(0.75)
      max = df[col].max()
      mean = df[col].mean()
      std = df[col].std()
      skewness = df[col].skew()
      kurtosis = df[col].kurtosis()

      output_df.loc[col] = [dtype, count, missing, unique, mode, min, Q1,
                            median, Q3, max, mean, std, skewness, kurtosis ]

      sns.histplot(data=df, x=col)
      plt.show()
    else:
      output_df.loc[col] = [dtype, count, missing, unique, mode, '-', '-',
                            '-', '-', '-', '-', '-', '-', '-']
      sns.countplot(data=df, x=col)
      plt.show()
  # output_df.to_csv('EDA.csv')
  return output_df

In [ ]:
def bivariate_stats(df, label, roundto=4):
  import pandas as pd
  from scipy import stats

  output_df = pd.DataFrame(columns=['p-value', 'r-value', 'y = m(x) + b', 'F', 'X2'])

  for feature in df:
    # There Might be missing values for which analysis can come up with inaccuracy
    if feature != label:
     if pd.api.types.is_numeric_dtype(df[feature]) and pd.api.types.is_numeric_dtype(df[label]):
       # Process N2N relationships
       m, b, r, p, err = stats.linregress(df[feature], df[label])
       output_df.loc[feature] = [round(p, roundto), round(r, roundto), f"y={round(m, roundto)}x+{round(b, roundto)}", '-', '-']
     elif not pd.api.types.is_numeric_dtype(df[feature]) and not pd.api.types.is_numeric_dtype(df[label]):
       #  Process C2C relationships
       contingency_table = pd.crosstab(df[feature], df[label])
       chi2, p, dof, expected = stats.chi2_contingency(contingency_table)

       output_df.loc[feature] = [round(p, roundto), '-', '-', '-', round(chi2, roundto)]
     else:
       # Process C2N & N2C relationships
       if pd.api.types.is_numeric_dtype(df[feature]):
        num = feature
        cat = label
       else:
        num = label
        cat = feature

       groups = df[cat].unique()
       group_lists = []
       for g in groups:
        group_lists.append(df[df[cat] == g][num])

       f, p = stats.f_oneway(*group_lists) # same as (group_lists[0], group_lists[1], ..., group_lists[n])

       output_df.loc[feature] = [round(p, roundto), '-', '-', round(f, roundto), '-']
  # return output_df.sort_values(by=['r-value'], ascending=False)
  #From strongest correlation to weakest correlation

  return output_df




In [ ]:
import kagglehub
import pandas as pd
# Download latest version
mental_heath_data = kagglehub.dataset_download("shakilh/bangladeshi-university-students-mental-health")
health_insurance_data = kagglehub.dataset_download("teertha/ushealthinsurancedataset")

df_insurance = pd.read_csv(health_insurance_data + "/insurance.csv")
# df = pd.read_csv(mental_heath_data + "/Raw Data.csv")

# univariate(df_insurance)
bivariate_stats(df_insurance, 'charges')



,p-value,r-value,y = m(x) + b,F,X2
age,0.0000,0.299,y=257.7226x+3165.885,-,-
sex,0.0361,-,-,4.3997,-
bmi,0.0000,0.1983,y=393.873x+1192.9372,-,-
children,0.0129,0.068,y=683.0894x+12522.4955,-,-
smoker,0.0000,-,-,2177.6149,-
region,0.0309,-,-,2.9696,-
